In [34]:
# !pip install pytorch_lightning

In [35]:
import os, random, math, pickle
import pandas as pd
import numpy as np
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from torch.utils.data import random_split
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

# Set environment variables for reproducibility and safety
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import precision_score, recall_score, accuracy_score

# # 1. Configuration & Seeding
# SEED = 42
# random.seed(SEED)
# np.random.seed(SEED)
# torch.manual_seed(SEED)
# if torch.cuda.is_available():
#     torch.cuda.manual_seed_all(SEED)

In [36]:
# name = 'book'
# N_STATIC_RELATION = 20
# N_CLUSTER = 15

# USER_START_ID = 1
# USER_END_ID = 2890
# ITEM_START_ID = 2891  # book: 2891
# ITEM_END_ID = 11584
# STATIC_ENTITY_START_ID = 11585  # book: 2891
# STATIC_ENTITY_END_ID = 38885

# N_ENTITY = STATIC_ENTITY_END_ID
# N_RELATION = N_STATIC_RELATION + N_CLUSTER

In [37]:
name = 'movie'
N_STATIC_RELATION = 20
N_CLUSTER = 15

USER_START_ID = 1
USER_END_ID = 1439
ITEM_START_ID = 1440  # book: 2891
ITEM_END_ID = 6335
STATIC_ENTITY_START_ID = 6336  # book: 2891
STATIC_ENTITY_END_ID = 54535

N_ENTITY = STATIC_ENTITY_END_ID
N_RELATION = N_STATIC_RELATION + N_CLUSTER

#### 1.1 KGCN Model

In [38]:
from collections import defaultdict

class KnownledgeGraph():
    def __init__(self, graph_df, nbr_sample_size=1):
        self.graph_df = graph_df
        self.graph_nbr_sample_size = nbr_sample_size
        self.graph_entities = None
        self.graph_relations = None
        self.graph = defaultdict(set)

    def build(self):
        df = self.graph_df

        # Build the knowledge graph with sets
        for head_id, relation_id, tail_id, _, _, _ in df.values:
            self.graph[head_id].add((tail_id, relation_id))
            self.graph[tail_id].add((head_id, relation_id))

    def get_neighbors(self, entity_ids, sample_size=8):
        """
        Lấy hàng xóm cho một batch các entity.
        entity_ids: Tensor kích thước [batch] hoặc một list các ID.
        """
        if sample_size is None:
            sample_size = self.graph_nbr_sample_size

        # 1. Chuyển tensor về list thuần Python để vòng lặp chạy ở tốc độ cao nhất
        if torch.is_tensor(entity_ids):
            entities = entity_ids.cpu().tolist()
        else:
            entities = entity_ids

        batch_adj_entities = []
        batch_adj_relations = []

        # 2. Xử lý từng entity trong batch
        for e in entities:
            # Lấy tập hàng xóm, trả về list rỗng nếu không tìm thấy
            neighbors = list(self.graph.get(e, set()))
            n_neighbors = len(neighbors)

            if n_neighbors == 0:
                # XỬ LÝ NGOẠI LỆ: Nếu không có hàng xóm, đệm bằng ID 0 cho đủ N
                # Lưu ý: ID 0 nên được quy ước là padding index trong hệ thống của bạn
                sampled_neighbors = [(0, 0)] * sample_size
                
            elif n_neighbors >= sample_size:
                # Số hàng xóm >= N: Lấy mẫu KHÔNG hoàn lại (Không trùng lặp)
                # random.sample chạy siêu nhanh với list
                sampled_neighbors = random.sample(neighbors, sample_size)
                
            else:
                # Số hàng xóm < N: Lấy mẫu CÓ hoàn lại (Chấp nhận trùng lặp để lấp đầy)
                # random.choices được tối ưu hóa bằng C dưới nền nên cực kỳ mượt
                sampled_neighbors = random.choices(neighbors, k=sample_size)

            # 3. Tách riêng entity và relation cho entity hiện tại
            # Mỗi list con sẽ có chính xác độ dài là sample_size (N)
            adj_e = [n for n, r in sampled_neighbors]
            adj_r = [r for n, r in sampled_neighbors]

            batch_adj_entities.append(adj_e)
            batch_adj_relations.append(adj_r)

        # 4. Gom tất cả list con thành ma trận 2D và chuyển thành Tensor
        # Kích thước đầu ra: [batch, sample_size]
        adj_entities_tensor = torch.tensor(batch_adj_entities, dtype=torch.long)
        adj_relations_tensor = torch.tensor(batch_adj_relations, dtype=torch.long)

        return adj_entities_tensor, adj_relations_tensor

In [39]:
class SumAggregator(nn.Module):
    def __init__(self, embedding_dim):
        super(SumAggregator, self).__init__()
        self.linear = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, neighbor_embs, central_embs):
        """
        neighbor_embs: Tensor of shape (batch, emb_dim)
        central_embs: Tensor of shape (batch, emb_dim)
        """

        # Combine with central entity embedding
        combined = neighbor_embs + central_embs  # shape: (batch, emb_dim) ([1204, 64])

        # Linear + activation
        output = torch.tanh(self.linear(combined))  # shape: (batch, emb_dim) torch.Size([1204, 64])
        return output

In [48]:
class KGCN(pl.LightningModule):
    def __init__(self, graph: KnownledgeGraph, item_pool, item2pool_idx, num_neg_samples, embedding_dim, lr, loss_margin, temperature ):
        super().__init__()
        self.save_hyperparameters()

        model_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model_device = model_device

        self.graph = graph
        self.item_pool = item_pool # Move item_pool to the correct device .to(model_device)
        self.item2pool_idx = item2pool_idx
        

        self.num_neg_samples = num_neg_samples

        self.embedding_dim = embedding_dim

        # Embedding layersnum_neg_samples
        self.entity_embedding = nn.Embedding(N_ENTITY + 1, embedding_dim, padding_idx = 0).to(model_device)
        self.relation_embedding = nn.Embedding(N_RELATION + 1, embedding_dim, padding_idx = 0).to(model_device)

        # Aggregator
        self.aggregator = SumAggregator(embedding_dim).to(model_device)

        self.dropout = nn.Dropout(p=0.1)
        self.loss_function = torch.nn.MarginRankingLoss(margin= loss_margin)
        self.temperature= temperature

    def setup(self, stage=None):
        self.train_user_pos_items = self.trainer.datamodule.train_user_pos_items
        self.val_user_pos_items = self.trainer.datamodule.val_user_pos_items

    @staticmethod
    def hit_at_k(pred_items, true_items, k):
        hits = 0
        for pred, true in zip(pred_items, true_items):
            # Count if any true item is in top-k predictions
            if len(set(pred[:k]) & set(true)) > 0:
                hits += 1
        return hits / len(true_items)

    @staticmethod
    def ndcg_at_k(pred_items, true_items, k):
        ndcg = 0.0
        for pred, true in zip(pred_items, true_items):
            gains = []
            for idx, item in enumerate(pred[:k]):
                gains.append(1 if item in true else 0)
            ideal_gains = [1] * min(len(true), k)
            dcg = sum(g / math.log2(i+2) for i, g in enumerate(gains))
            idcg = sum(g / math.log2(i+2) for i, g in enumerate(ideal_gains))
            ndcg += dcg / idcg if idcg > 0 else 0
        return ndcg / len(true_items)

    @staticmethod
    def recall_at_k(pred_items, true_items, k):
        recall = 0.0
        for pred, true in zip(pred_items, true_items):
            recall += len(set(pred[:k]) & set(true)) / len(true)
        return recall / len(true_items)

    @staticmethod
    def precision_at_k(pred_items, true_items, k):
        precision = 0.0
        for pred, true in zip(pred_items, true_items):
            precision += len(set(pred[:k]) & set(true)) / k
        return precision / len(true_items)

    def enhance_item_embs(self, user_ids, item_ids):
        '''
        user_ids: (batch)
        item_ids: (batch). Mỗi phần tử trong batch chỉ 1 item.
        '''

        neighbor_ids, relation_ids = self.graph.get_neighbors(item_ids) # (batch, N). Mỗi phần tử trong batch có nhiều hơn 1 neighbor

        user_embs = self.entity_embedding(user_ids) # (batch, embed_dim)
        item_embs = self.entity_embedding(item_ids) # (batch, embed_dim) 

        neighbor_embs = self.entity_embedding(neighbor_ids) # (batch, N, embed_dim)
        relation_embs = self.relation_embedding(relation_ids) # (batch, N, embed_dim)

        ################ User - Relation Attention ################
        # Expand user_embs to match neighbor dimension
        ##############################
        user_embs_norm = F.normalize(user_embs, p=2, dim=1) # (batch, embed_dim)
        relation_embs_norm = F.normalize(relation_embs, p=2, dim=2) # (batch, embed_dim)
        scores = (user_embs_norm.unsqueeze(1) * relation_embs_norm).sum(dim=2) # (batch, N)
        ##############################
        # user_embs_expanded = user_embs_expanded = user_embs.unsqueeze(1) 
        # scores = torch.exp(-torch.abs(user_embs_expanded - relation_embs).sum(dim=2))

        attention_weights = F.softmax(scores, dim=1).unsqueeze(-1)  # [1204, 8, 1]
        weighted_neighbor_embs = neighbor_embs * attention_weights  # [1204, 8, 64]
        weighted_neighbor_embs = weighted_neighbor_embs.mean(dim=1) # [1204, 64]
        ################ User - Relation Attention ################

        updated_item_embs  = self.aggregator(weighted_neighbor_embs, item_embs) # (batch, embed_dim)

        return user_embs, updated_item_embs

    def enhance_multi_item_embs(self, user_ids, multi_item_ids):
        '''
        user_ids: (batch,)
        multi_item_ids: (batch, N)
        '''
        batch_size, N = multi_item_ids.shape
        
        enhanced_items_list = []
        user_embs_final = None
        
        # Chạy vòng lặp N lần, mỗi lần xử lý 1 cột item cho toàn bộ batch user
        for i in range(N):
            # Lấy ra cột thứ i (cột này chứa item thứ i của tất cả user trong batch)
            # Shape của current_item_ids sẽ là (batch,) - khớp hoàn toàn với hàm cũ
            current_item_ids = multi_item_ids[:, i] # torch.Size([512])
            
            # Đưa qua hàm enhance nguyên bản (chạy với batch size 512 cực nhẹ nhàng)
            u_embs, updated_item_embs = self.enhance_item_embs(user_ids, current_item_ids)
            
            # Vì user_ids không thay đổi qua các vòng lặp, ta chỉ cần lưu user_embs 1 lần
            if i == 0:
                user_embs_final = u_embs
                
            # Cất kết quả của item vào danh sách
            # updated_item_embs có shape: (batch, embed_dim)
            enhanced_items_list.append(updated_item_embs)

        # Xếp chồng N tensor lại với nhau dọc theo chiều thứ 1 (dim=1)
        # Đầu vào: 1 list chứa N tensor kích thước (batch, embed_dim)
        # Đầu ra: 1 tensor 3D kích thước (batch, N, embed_dim)
        updated_multi_item_embs = torch.stack(enhanced_items_list, dim=1) #torch.Size([512, 1023, 64])

        return user_embs_final, updated_multi_item_embs

    def forward(self, batch):
        '''
        user_ids: (batch)
        item_ids: (batch). Mỗi phần tử trong batch chỉ 1 item.
        '''
        user_ids, item_ids = batch

        user_embs, updated_item_embs = self.enhance_item_embs(user_ids, item_ids)

        return user_embs, updated_item_embs

    def compute_loss(self, batch, user_embs, updated_item_embs):
        user_ids, item_ids = batch

        pos_item_embs = updated_item_embs

        # ###############
        user_embs_norm = F.normalize(user_embs, p=2, dim=1) # (batch, embed_dim)
        pos_item_embs_norm = F.normalize(pos_item_embs, p=2, dim=1) # (batch, embed_dim)
        pos_scores = (user_embs_norm * pos_item_embs_norm).sum(dim=1) # (batch,)
        # ###############
        # pos_scores = torch.exp(-torch.abs(user_embs - pos_item_embs).sum(dim=1))

        ########################## Random sample (batch, self.num_sample_items, embedd_dim) ##################
        users = user_ids.cpu().tolist()
        pool_size = len(self.item_pool)
        neg_item_ids_list = []

        # 2. Lấy mẫu âm có kiểm tra loại trừ cho từng user
        for u in users:
            # ÉP SANG SET NGAY TẠI ĐÂY: Chuyển list thành set để phép kiểm tra 'not in' chạy với tốc độ O(1)
            pos_items_of_u = set(self.train_user_pos_items[u])

            user_negs = []
            # Tiếp tục bốc ngẫu nhiên cho đến khi đủ số lượng num_neg_samples
            while len(user_negs) < self.num_neg_samples:
                rand_idx = random.randint(0, pool_size - 1)

                # Nếu item_pool là Tensor, nhớ dùng .item() để lấy số nguyên thuần Python
                sampled_item = self.item_pool[rand_idx].item()

                # CHỐT CHẶN SIÊU TỐC: Chỉ chấp nhận nếu item vừa bốc KHÔNG nằm trong set
                if sampled_item not in pos_items_of_u:
                    user_negs.append(sampled_item)

            neg_item_ids_list.append(user_negs)

        # 3. Chuyển đổi list kết quả (batch_size, num_neg_samples) thành Tensor và đẩy thẳng lên GPU
        neg_item_ids = torch.tensor(neg_item_ids_list, dtype=torch.long, device=self.model_device) # torch.Size([512, n_sample 50])

        # 4. Lấy embeddings (Lưu ý: Không dùng F.normalize ở đây như chúng ta đã thống nhất)
        # neg_item_embs = self.entity_embedding(neg_item_ids) # torch.Size([512, n sample 50, dim 64])
        _, neg_item_embs = self.enhance_multi_item_embs(user_ids, neg_item_ids)


        # 5. Tính điểm Dot Product cho toàn bộ mẫu âm (Shape: [batch_size, num_neg_samples])
        ###############
        neg_item_embs_norm = F.normalize(neg_item_embs, p=2, dim=2)
        neg_scores = (user_embs_norm.unsqueeze(1) * neg_item_embs_norm).sum(dim=2) # torch.Size([512, n_sample 50])
        ###############
        # user_embs_expanded = user_embs.unsqueeze(1) 
        # neg_scores = torch.exp(-torch.abs(user_embs_expanded - neg_item_embs).sum(dim=2))#torch.Size([512, n_sample 50])


        # 6. Chọn số lượng hard negatives (Ví dụ: từ 20 mẫu âm bốc ra, chọn 5 mẫu khó nhất)
        k = 10

        # Lấy trực tiếp VALUE (điểm số cao nhất) thay vì INDICES
        # Shape của hard_neg_scores: [batch_size, k]
        hard_neg_scores = torch.topk(neg_scores, k=k, dim=1).values # # torch.Size([512, 10)

        # 7. Tính trung bình điểm của các hard negatives này
        # Shape cuối cùng: [batch_size] -> Sẵn sàng đưa vào hàm Loss
        neg_scores = hard_neg_scores.mean(dim=1) #torch.Size([512])
        ########################## Random sample (batch, self.num_sample_items, embedd_dim) ##################

        # ####################### Compute Loss #######################
        # scores = torch.cat([pos_scores, neg_scores], dim=0)
        # labels = torch.cat([torch.ones_like(pos_scores), torch.zeros_like(neg_scores)], dim=0)

        # # Use binary_cross_entropy_with_logits instead of binary_cross_entropy
        # # because scores are raw logits (dot products), not probabilities (0-1)
        # # loss = F.binary_cross_entropy_with_logits(scores, labels)
        # loss = F.binary_cross_entropy(scores, labels)

        loss = - F.logsigmoid(pos_scores - neg_scores).mean() #### chay duoc

        # margin = 0.5
        # loss = F.relu(neg_scores - pos_scores + margin).mean()

        # loss = self.loss_function(pos_scores, neg_scores, target=torch.ones_like(pos_scores))
        ####################### Compute Loss #######################

        return loss, pos_scores, neg_scores

    def training_step(self, batch, batch_idx):
        user_embs, updated_item_embs = self(batch)
        train_loss, ps, ns = self.compute_loss(batch, user_embs, updated_item_embs)

        self.log("train_loss", train_loss, on_epoch=True, prog_bar=True)

        return train_loss

    def validation_step(self, batch, batch_idx):
        user_ids, item_ids = batch

        user_embs, updated_item_embs = self(batch)

        val_loss, ps, ns = self.compute_loss(batch, user_embs, updated_item_embs)
        self.log('val_loss', val_loss, prog_bar=True, logger=True)

        # ################ Calculate metrics

        # # 1. Lấy kích thước batch thực tế từ user_ids
        # batch_size = user_ids.size(0)

        # # 2. Phù phép item_pool từ 1D thành 2D
        # # - unsqueeze(0): Biến [N] thành [1, N]
        # # - expand(batch_size, -1): Kéo dãn chiều 1 thành chiều batch
        # multi_item_ids = self.item_pool.unsqueeze(0).expand(batch_size, -1)  # torch.Size([512, 1023]) # self.item_pool shape = (N_item_pool), multi_item_ids = (batch, N_item_pool)
        # _, updated_pool_item_embs = self.enhance_multi_item_embs(user_ids, multi_item_ids)


        # user_embs_norm = F.normalize(user_embs, p=2, dim=1) # torch.Size([512, 64])
        # updated_pool_item_embs_norm = F.normalize(updated_pool_item_embs, p=2, dim=2) # torch.Size([N = 1023, 64]), #torch.Size([512, 1023, 64])
        
                
        # # # Tính scores (Shape: [batch_size, num_items_in_pool])
        # ##### Cach 1
        # # scores = (user_embs_norm.unsqueeze(1) * updated_pool_item_embs_norm).sum(dim=2) # torch.Size([512, 1023])

        # #### Cach 2
        # # 1. Biến user_embs thành ma trận hàng: (batch, dim) -> (batch, 1, dim)
        # user_embs_norm_expanded = user_embs_norm.unsqueeze(1)
        # # 2. Đảo chiều ma trận item: (batch, N, dim) -> (batch, dim, N)
        # # Lưu ý: Dùng .transpose(1, 2) chứ KHÔNG dùng .T vì .T sẽ lật cả chiều batch
        # i_transposed = updated_pool_item_embs_norm.transpose(1, 2)
        # # 3. Nhân ma trận: (batch, 1, dim) @ (batch, dim, N) = (batch, 1, N)
        # # Sau đó dùng .squeeze(1) để ép bẹp chiều 1, kết quả ra đúng (batch, N)
        # scores = torch.bmm(user_embs_norm_expanded, i_transposed).squeeze(1)

        # # ==========================================
        # # MASKING SIÊU TỐC VỚI ADVANCED INDEXING
        # # ==========================================
        # # 1. Gom tất cả các tọa độ cần Mask vào 2 list
        # for i, u in enumerate(user_ids.tolist()):
        #     trained_items = self.train_user_pos_items[u]
        #     cols = []
        #     for item in trained_items:
        #         item_id = self.item2pool_idx[item]
        #         cols.append(item_id) # Cột tương ứng của item đó
        #         scores[i, cols] = float('-inf')       

        # pool_ids_tensor = torch.tensor(self.item_pool, dtype=torch.long, device=self.device) # shape (N_item_pool)
        
        # k_values = [10]  # Example: you can add more values as needed

        # for k in k_values:
        #     topk_relative_indices = torch.topk(scores, k=k, dim=1).indices

        #     # 4. Map ngược ra ID thực tế bằng cách truyền thẳng vào pool_ids_tensor
        #     # Lưu ý: pool_ids_tensor chính là torch.tensor(self.item_pool) mà bạn đã tạo ở trên
        #     # Shape: [batch_size, k]
        #     topk_actual_ids = pool_ids_tensor[topk_relative_indices]

        #     # 5. (Tùy chọn) Ép về Python list dạng [ [id1, id2...], [id3, id4...] ]
        #     # Để đưa vào các hàm tính Hit@10 hoặc NDCG@10
        #     topk_items = topk_actual_ids.tolist()

        #     true_items = []  # (1024, variable length), each user may have multiple positive items
        #     for u in user_ids.tolist():
        #         adjusted_val_items = [item for item in self.val_user_pos_items[u]]
        #         true_items.append(adjusted_val_items)


        #     # Compute metrics for this k
        #     hit = self.hit_at_k(topk_items, true_items, k)
        #     ndcg = self.ndcg_at_k(topk_items, true_items, k)
        #     recall = self.recall_at_k(topk_items, true_items, k)
        #     precision = self.precision_at_k(topk_items, true_items, k)

        #     # Log metrics dynamically
        #     self.log(f"val_hit@{k:02d}", hit, prog_bar=True)
        #     self.log(f"val_recall@{k:02d}", recall, prog_bar=True)
        #     self.log(f"val_precision@{k:02d}", precision, prog_bar=True)
        #     self.log(f"val_ndcg@{k:02d}", ndcg, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr, weight_decay=5e-4)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.75)
        return [optimizer], [scheduler]

#### 1.2 DataModule

In [41]:
class RecommenderDataModule(pl.LightningDataModule):
    def __init__(self, interaction_df, graph: KnownledgeGraph, batch_size, val_size):
        super().__init__()
        self.interaction_df = interaction_df
        self.batch_size = batch_size
        self.val_size = val_size
        self.graph = graph

    def prepare_data(self):
        # Load interactions
        df_inter = self.interaction_df

        # Build positive interactions
        dataset = []    # user_id, entity_id (item_id), neighbor_ids, relation_ids,
        for _, row in df_inter.iterrows():
            u = row['user_id']
            e = row['entity_id']
            dataset.append((u, e))

        # Split interactions for validation
        train_size = int(len(dataset) * (1 - self.val_size))
        val_size = len(dataset) - train_size
        self.train_dataset, self.val_dataset = random_split(dataset, [train_size, val_size])

        train_user_pos_items = defaultdict(set)
        for u, i in self.train_dataset:
            train_user_pos_items[u].add(i)

        val_user_pos_items = defaultdict(set)
        for u, i, in self.val_dataset:
            val_user_pos_items[u].add(i)

        self.train_user_pos_items = train_user_pos_items
        self.val_user_pos_items = val_user_pos_items

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False)

#### 1.4 Main

In [49]:
tckg_df = pd.read_csv(f'./data/{name}_TCKG.csv')
graph_df = tckg_df[tckg_df['relation_id'] <= 20]

interaction_df = tckg_df[tckg_df['relation_id'] >=21]

interaction_df = interaction_df.rename(columns={'head_id': 'user_id', 'tail_id': 'entity_id'})
unique_items = interaction_df['entity_id'].unique()
item_pool = torch.tensor(unique_items, dtype=torch.long)
item2pool_idx = {item_id: idx for idx, item_id in enumerate(unique_items)}

graph = KnownledgeGraph(graph_df)
graph.build()

data_module = RecommenderDataModule(interaction_df, graph, batch_size=512, val_size=0.1)
data_module.prepare_data()


In [ ]:
from pytorch_lightning import Trainer

model = KGCN(
        graph = graph,
        item_pool = item_pool,# Shape: [batch] -> [batch, 1] -> [batch, N]
        item2pool_idx= item2pool_idx,
        num_neg_samples = 20,
        embedding_dim= 64,
        lr= 0.01,
        loss_margin = 0.5,
        temperature = 1
    )

early_stop_callback = EarlyStopping(
    monitor="val_loss",     # metric to monitor
    patience=50,            # number of epochs wi# Shape: [batch] -> [batch, 1] -> [batch, N]th no improvement after which training will be stopped
    mode="min",             # 'min' because we want to minimize val_loss
    verbose=True
)

checkpoint_hit = ModelCheckpoint(
        monitor="val_loss",
        dirpath="./checkpoints",
        filename=f"{name}-{{epoch:02d}}-{{val_loss:.3f}}",
        save_top_k=1,
        mode="min",
)# Shape: [batch] -> [batch, 1] -> [batch, N]

trainer = Trainer(
        num_sanity_val_steps=0,
        max_epochs=50,
        accelerator="auto",
        # limit_val_batches=0, # 2. CHỐT CHẶN: Ép số lượng batch validation chạy trong mỗi epoch về con số 0
        callbacks=[checkpoint_hit, early_stop_callback],
    )

# trainer.fit(model, data_module)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [52]:
path = r"C:\Users\vtruong1\OneDrive - Intel Corporation\Desktop\Van Hao\Study\21. Luan Van\03. Learn Embedding\checkpoints\movie-epoch=33-val_loss=0.586.ckpt"
model = KGCN.load_from_checkpoint(path, weights_only= False)



# 2. Khởi tạo một Trainer mới "nhẹ nhàng"
# Lúc này bạn không cần max_epochs hay các callbacks phức tạp như lúc train nữa
trainer = Trainer(    
        num_sanity_val_steps=0,
        max_epochs=50,
        accelerator="auto",
        # limit_val_batches=0, # 2. CHỐT CHẶN: Ép số lượng batch validation chạy trong mỗi epoch về con số 0
        callbacks=[checkpoint_hit, early_stop_callback],
)

# 3. Kích hoạt chạy Validation
# Truyền trực tiếp model và data_module vào, Lightning sẽ tự động tìm val_dataloader
trainer.fit(model, datamodule=data_module)
# In kết quả ra màn hình
# print(results)

# (Tùy chọn) Đảm bảo mô hình ở chế độ evaluation để tắt Dropout/BatchNorm
# model.eval() 
# results = trainer.validate(model, datamodule=data_module)
# print(results)



GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name               | Type              | Params | Mode 
-----------------------------------------------------------------
0 | entity_embedding   | Embedding         | 3.5 M  | train
1 | relation_embedding | Embedding         | 2.3 K  | train
2 | aggregator         | SumAggregator     | 4.2 K  | train
3 | dropout            | Dropout           | 0      | train
4 | loss_function      | MarginRankingLoss | 0      | train
-----------------------------------------------------------------
3.5 M     Trainable params
0         Non-trainable params
3.5 M     Total params
13.987    Total estimated model params size (MB)
6         Modules in train mode
0         Modules in eval mode


Epoch 29:  85%|████████▌ | 35/41 [00:16<00:02,  2.14it/s, v_num=41, train_loss_step=0.558, val_loss=0.588, train_loss_epoch=0.546]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

In [45]:
# 1. Tạo tensor có chiều Batch và đẩy lên đúng Device
user_id = torch.tensor([1236], device=model.device)
item_id = torch.tensor([4768], device=model.device)

# # 2. Đảm bảo mô hình ở chế độ đánh giá (Tắt Dropout)
# model.eval()

# 3. Lấy Embedding (Tắt đạo hàm để chạy nhanh và không tốn RAM)
with torch.no_grad():
    user_embs, updated_item_embs = model.enhance_item_embs(user_id, item_id)
    # print(user_embs)
    # print(updated_item_embs)



    user_embs_norm = F.normalize(user_embs, p=2, dim=1) # (batch, embed_dim)
    updated_item_embs_norm = F.normalize(updated_item_embs, p=2, dim=1) # (batch, embed_dim)
    scores = (user_embs_norm * updated_item_embs_norm).sum(dim=1) # (batch,)
    scores = (1 + scores)

    # scores = torch.exp(-torch.abs(user_embs - updated_item_embs).sum(dim=1))


# In ra điểm số dạng số thực Python
print(f"Điểm dự đoán: {scores.item():.4f}")

Điểm dự đoán: 0.4223
